# 12. 실습실 9B QLoRA 학습

담당: 유림
목표: 9B 모델의 preflight를 통과시키고, 저장과 재개가 실제로 동작하는지 확인한 뒤, 본 학습을
분리된 프로세스로 띄워서 밤새 돌아가게 만드는 것입니다.

가장 중요한 단계는 3번의 중단과 재개 시험입니다. 여기를 건너뛰면 밤중에 세션이 끊겼을 때
처음부터 다시 학습해야 하는 상황이 생깁니다.

## 선행 조건
- 수빈의 split이 확정되어 있어야 합니다.
- 창혁의 `artifacts/input_config_frozen.json` 이 있으면 그 값에 맞춰 config를 덮어씁니다.

## 통과 기준(PASS)
1. preflight가 20 optimizer step과 50문항 추론을 통과하고 peak VRAM이 16GB 안에 들어옵니다.
2. 20-step smoke 학습이 끝나고 checkpoint가 생성됩니다.
3. 중단한 뒤 같은 명령을 다시 실행했을 때 저장된 step부터 이어지고, loss가 튀지 않습니다.
4. 본 학습 job이 살아 있고 `train_log.jsonl` 에 기록이 쌓입니다.

## 넘겨줄 내용
adapter 경로, 학습 step 수, validation 정확도를 준혁과 수정에게 전달합니다.

### 명령 인자에 대한 안내

이 노트북의 명령은 `vqa/` 와 `scripts/` 에 실제로 구현된 인자에 맞춰 작성했습니다. 그래도 코어
코드가 계속 바뀌는 동안에는 인자 이름이 달라질 수 있습니다. 셀이 `unrecognized arguments` 로
실패하면 먼저 해당 명령에 `--help` 를 붙여서 실제 인자 이름을 확인하고, 셀 안의 문자열만 고치면
됩니다. 노트북의 다른 부분은 고치지 않아도 됩니다. 고친 내용은 팀 채널에 알려서 다른 사람이 같은
곳에서 막히지 않게 하십시오.

`python -m vqa.jobs start` 의 `--auto-restart` 는 재시작 횟수를 받는 숫자 인자입니다. 아래
`AUTO_RESTART` 를 0으로 두면 자동 재시작 없이 실행합니다. 재시작은 비정상 종료에만 적용되며,
정상적으로 끝난 작업을 다시 돌리지는 않습니다.

In [ ]:
# ===== 파라미터 =====
PROJECT_ROOT = r"."
DATA_ROOT = r"."
RUN_ROOT = r"./artifacts/runs"
LOCAL_SCRATCH = r"./artifacts/scratch"
MODEL_ROOT = r""

CONFIG = "configs/lab_qwen35_9b.yaml"
SPLIT_FILE = "artifacts/data/split_fold0.json"   # 팀 공식 학습 분할 (train_fold1234, 5,358문항)
VAL_IDS_FILE = "fold_csv_files/val_fold0.csv"    # 팀 공식 검증 분할 (fold 0, 1,344문항)
VAL_OUT_NAME = "fold0_ft"
FROZEN_INPUT = "artifacts/input_config_frozen.json"
FROZEN_KEY = "lab_qwen35_9b"

SMOKE_RUN_ID = "lab9b_smoke"
RESUME_RUN_ID = "lab9b_resume_test"
TRAIN_RUN_ID = "lab9b_ft_v1"

SMOKE_STEPS = 20
SMOKE_LIMIT_TRAIN = 200
RESUME_TEST_STEPS = 60      # 절반쯤에서 끊었다가 이어 붙일 목표 step
RESUME_TEST_WAIT_SEC = 180  # job을 띄운 뒤 강제로 중단하기까지 기다릴 시간

AUTO_RESTART = 3            # 비정상 종료 시 다시 띄울 횟수입니다. 0이면 재시작하지 않습니다.

DEV_TEACHER_RUN_ID = "lab9b_dev_teacher"
DEV_SHARD = "1/4"           # teacher 추론을 여러 PC로 나눌 때 각자 다르게 지정합니다.

In [ ]:
# --- 공통 부트스트랩: 경로, 환경 변수, 명령 실행 헬퍼 (수정할 일 거의 없음) ---
import os, sys, json, shlex, subprocess, time, csv, hashlib, random
from pathlib import Path

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def _setenv(name, value, create=False):
    if not value:
        return None
    p = Path(value).expanduser()
    if create:
        p.mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(p)
    return p


DATA_ROOT_P = _setenv("VQA_DATA_ROOT", DATA_ROOT)
RUN_ROOT_P = _setenv("VQA_RUN_ROOT", RUN_ROOT, create=True)
SCRATCH_P = _setenv("VQA_LOCAL_SCRATCH", LOCAL_SCRATCH, create=True)
MODEL_ROOT_P = _setenv("VQA_MODEL_ROOT", MODEL_ROOT)

PY = sys.executable


def run(cmd, check=True, echo=True, quiet=False, hint=True):
    """명령을 실행하면서 출력을 그대로 보여 주고 종료 코드와 소요 시간을 돌려줍니다.

    quiet가 True이면 명령의 출력도, 뒤에 붙는 "[rc=...]" 줄도 찍지 않습니다. 상태를 조회하는
    호출은 실패가 정상적인 결과입니다. 예를 들어 `vqa.jobs status` 는 죽은 job에 대해 종료 코드
    1을 돌려줍니다. 그 줄을 그대로 찍으면 셀 출력 맨 앞에 "[rc=1]"이 남아서, 실제 실패로 읽히는
    오해를 부릅니다.
    """
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    cmd = [str(x) for x in cmd]
    if echo:
        print("$ " + " ".join(shlex.quote(x) for x in cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        cmd, cwd=str(PROJECT_ROOT), stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8",
        errors="replace", bufsize=1,
    )
    buf = []
    for line in proc.stdout:
        buf.append(line)
        if not quiet:
            print(line, end="", flush=True)
    rc = proc.wait()
    dt = time.time() - t0
    if not quiet:
        print("[rc=%d] %.1f초" % (rc, dt), flush=True)
    if rc != 0:
        if hint:
            print("실패했습니다. 같은 명령에 --help 를 붙여서 인자 이름부터 확인하십시오.", flush=True)
        if check:
            raise RuntimeError("명령이 실패했습니다: " + " ".join(cmd))
    return {"rc": rc, "seconds": dt, "output": "".join(buf)}


def vqa(module, *args, **kw):
    """python -m vqa.<module> 을 실행합니다."""
    return run([PY, "-m", "vqa." + module] + [str(a) for a in args], **kw)


def script(name, *args, **kw):
    """scripts/<name> 을 실행합니다. name 에 / 가 있으면 저장소 루트 기준 경로로 봅니다."""
    path = Path(name) if "/" in name else Path("scripts") / name
    return run([PY, str(path)] + [str(a) for a in args], **kw)


def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))


def read_ids(path):
    return [ln.strip() for ln in Path(path).read_text(encoding="utf-8").splitlines() if ln.strip()]


def sha256_of(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    return digest[:limit] if limit else digest


print("PROJECT_ROOT      :", PROJECT_ROOT)
for _name in ("VQA_DATA_ROOT", "VQA_RUN_ROOT", "VQA_LOCAL_SCRATCH", "VQA_MODEL_ROOT"):
    print("%-18s: %s" % (_name, os.environ.get(_name, "(미설정)")))

In [ ]:
OVERRIDES = []
if Path(FROZEN_INPUT).exists():
    frozen = json.loads(Path(FROZEN_INPUT).read_text(encoding="utf-8"))
    OVERRIDES += ["--set", "prompt_id=%s" % frozen["prompt_id"]]
    mp = frozen.get("max_pixels", {}).get(FROZEN_KEY)
    if mp:
        OVERRIDES += ["--set", "max_pixels=%d" % mp]
    print("창혁이 동결한 입력 설정을 적용합니다:", OVERRIDES)
else:
    print("아직 동결된 입력 설정이 없습니다. config 기본값으로 진행합니다.")

JOB_EXTRA = ["--auto-restart", str(AUTO_RESTART)] if AUTO_RESTART else []
print("job 추가 옵션:", JOB_EXTRA or "(없음)")

## 1. preflight

계획서 3절의 항목을 한 번에 확인합니다. 단일 토큰 매핑, chat template 꼬리, 정답 마스킹, LoRA 대상
목록, 가장 긴 표본에서의 역전파, 속도, peak VRAM이 모두 여기에서 나옵니다.

출력에서 LoRA 대상 목록이 언어 모델 쪽 모듈만 포함하는지 직접 확인하십시오. vision 쪽에 이름이
같은 projection이 섞이면 학습 대상이 잘못 잡힙니다.

In [ ]:
vqa("preflight",
    "--config", CONFIG,
    "--n-train-steps", 20,
    "--n-infer", 50,
    *OVERRIDES)

## 2. 20-step smoke 학습

적은 데이터로 학습 경로 전체가 동작하는지 확인합니다. 여기에서 만들어진 adapter는 사용하지 않고
버립니다. 본 학습은 seed를 다시 설정한 새 adapter로 시작합니다.

In [ ]:
vqa("train",
    "--config", CONFIG,
    "--run-id", SMOKE_RUN_ID,
    "--split", SPLIT_FILE,
    "--max-steps", SMOKE_STEPS,
    "--limit-train", SMOKE_LIMIT_TRAIN,
    "--resume", "none",
    *OVERRIDES)

smoke_ckpt = RUN_ROOT_P / SMOKE_RUN_ID / "checkpoints"
print()
print("생성된 checkpoint:")
for p in sorted(smoke_ckpt.glob("step_*")):
    complete = (p / "COMPLETE").exists()
    print("  %s  COMPLETE=%s" % (p.name, complete))

## 3. 중단과 재개 시험 (가장 중요한 단계)

실제로 한 번 끊어 보고 이어지는지 확인합니다. 절차는 다음과 같습니다.

1. 분리된 job으로 학습을 시작합니다.
2. 몇 분 기다린 뒤 `jobs stop` 으로 강제 종료합니다.
3. 같은 시작 명령을 다시 실행합니다.
4. `train_log.jsonl` 에서 재개 지점의 step이 이어지고 loss가 매끄럽게 연결되는지 확인합니다.

재개 후 loss가 크게 튀면 optimizer 상태나 scheduler, RNG 중 하나가 복원되지 않은 것입니다.
그 상태로 본 학습을 시작하면 안 됩니다.

In [ ]:
def ascii_curve(xs, ys, width=64, height=12):
    """matplotlib 없이도 loss 추이를 확인할 수 있도록 문자로 곡선을 그립니다."""
    if not ys:
        return "(표시할 데이터가 없습니다.)"
    lo, hi = min(ys), max(ys)
    if hi - lo < 1e-9:
        hi = lo + 1e-9
    n = len(ys)
    grid = [[" "] * width for _ in range(height)]
    for i in range(width):
        j = int(i * (n - 1) / max(width - 1, 1))
        r = int((hi - ys[j]) / (hi - lo) * (height - 1))
        grid[r][i] = "*"
    out = []
    for r, row in enumerate(grid):
        v = hi - (hi - lo) * r / (height - 1)
        out.append("%9.4f |%s" % (v, "".join(row)))
    out.append(" " * 10 + "+" + "-" * width)
    out.append(" " * 11 + "step %s -> %s (%d개 기록)" % (xs[0], xs[-1], n))
    return "\n".join(out)


def read_train_log(run_id):
    path = RUN_ROOT_P / run_id / "train_log.jsonl"
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except ValueError:
            continue
    return rows


def log_summary(run_id, tail=8):
    rows = read_train_log(run_id)
    if not rows:
        print("아직 학습 기록이 없습니다:", run_id)
        return rows
    steps = [r.get("step") for r in rows if r.get("step") is not None]
    losses = [r.get("loss") for r in rows if isinstance(r.get("loss"), (int, float))]
    print("기록 %d줄 / step %s ~ %s" % (len(rows), steps[0] if steps else "?", steps[-1] if steps else "?"))
    for r in rows[-tail:]:
        print("  step=%s loss=%s lr=%s" % (r.get("step"), r.get("loss"), r.get("lr")))
    if losses:
        print(ascii_curve(steps[: len(losses)] or list(range(len(losses))), losses))
    return rows


RESUME_TRAIN_CMD = [PY, "-m", "vqa.train",
                    "--config", CONFIG,
                    "--run-id", RESUME_RUN_ID,
                    "--split", SPLIT_FILE,
                    "--max-steps", str(RESUME_TEST_STEPS),
                    "--resume", "auto"] + OVERRIDES
print("재개 시험에 사용할 명령:")
print("  " + " ".join(RESUME_TRAIN_CMD))

In [ ]:
# (1) 분리된 job으로 학습 시작
run([PY, "-m", "vqa.jobs", "start", "--name", RESUME_RUN_ID, "--"] + RESUME_TRAIN_CMD)

In [ ]:
# (2) 몇 분 기다렸다가 상태를 확인하고 강제로 중단합니다.
time.sleep(RESUME_TEST_WAIT_SEC)
run([PY, "-m", "vqa.jobs", "status", "--name", RESUME_RUN_ID], check=False)
run([PY, "-m", "vqa.jobs", "tail", "--name", RESUME_RUN_ID], check=False)

before = log_summary(RESUME_RUN_ID)
STEP_BEFORE_STOP = max([r.get("step", 0) for r in before] or [0])
print()
print("중단 직전 step:", STEP_BEFORE_STOP)

run([PY, "-m", "vqa.jobs", "stop", "--name", RESUME_RUN_ID], check=False)
time.sleep(10)
run([PY, "-m", "vqa.jobs", "status", "--name", RESUME_RUN_ID], check=False)

In [ ]:
# (3) 똑같은 시작 명령을 다시 실행합니다. 수정하는 부분은 없습니다.
run([PY, "-m", "vqa.jobs", "start", "--name", RESUME_RUN_ID, "--"] + RESUME_TRAIN_CMD)
time.sleep(RESUME_TEST_WAIT_SEC)
run([PY, "-m", "vqa.jobs", "status", "--name", RESUME_RUN_ID], check=False)

In [ ]:
# (4) 재개 지점을 확인합니다.
after = log_summary(RESUME_RUN_ID)
steps_after = [r.get("step", 0) for r in after]
STEP_AFTER = max(steps_after or [0])

print()
print("중단 직전 step: %s" % STEP_BEFORE_STOP)
print("재개 후 step  : %s" % STEP_AFTER)

resumed_from_scratch = len([s for s in steps_after if s and s <= 1]) > 1
if STEP_AFTER > STEP_BEFORE_STOP and not resumed_from_scratch:
    print("PASS: 저장된 step부터 이어졌습니다.")
else:
    print("확인 필요: 처음부터 다시 시작했을 가능성이 있습니다. checkpoints/latest.json을 확인하십시오.")

losses = [r.get("loss") for r in after if isinstance(r.get("loss"), (int, float))]
if len(losses) > 4:
    print()
    print("loss 추이가 재개 지점에서 급격히 튀지 않는지 위 곡선에서 확인하십시오.")

## 4. 본 학습 시작

`--resume auto` 가 기본이므로, 이 셀을 몇 번 다시 실행해도 처음부터 학습하지 않습니다. 분리된
프로세스로 띄우기 때문에 노트북을 닫아도 계속 돌아갑니다. 다만 PC가 꺼지거나 로그아웃되면
중단되므로, 퇴실 전에 실습실의 전원 정책을 반드시 확인하십시오.

In [ ]:
TRAIN_CMD = [PY, "-m", "vqa.train",
             "--config", CONFIG,
             "--run-id", TRAIN_RUN_ID,
             "--split", SPLIT_FILE,
             "--resume", "auto"] + OVERRIDES

run([PY, "-m", "vqa.jobs", "start", "--name", TRAIN_RUN_ID] + JOB_EXTRA + ["--"] + TRAIN_CMD)
print()
print("학습 job을 띄웠습니다. 아래 감시 셀로 진행 상황을 확인하십시오.")
print("로그 파일:", RUN_ROOT_P / "jobs" / (TRAIN_RUN_ID + ".log"))

## 5. 감시

이 셀은 여러 번 실행해도 안전합니다. 상태, 최근 로그, loss 곡선을 한 번에 보여 줍니다.

In [ ]:
run([PY, "-m", "vqa.jobs", "status", "--name", TRAIN_RUN_ID], check=False)
run([PY, "-m", "vqa.jobs", "tail", "--name", TRAIN_RUN_ID], check=False)
print()
log_summary(TRAIN_RUN_ID, tail=10)

latest = RUN_ROOT_P / TRAIN_RUN_ID / "checkpoints" / "latest.json"
if latest.exists():
    print()
    print("latest.json:", latest.read_text(encoding="utf-8")[:400])

## 6. 학습 뒤 validation 평가

학습이 끝나면 `final_adapter` 로 고정 validation을 추론하고 정확도를 확인합니다. base보다 낮으면
그 사실을 그대로 보고하십시오. 계획서 8절에 따라 무조건 FT를 제출하지는 않습니다.

In [ ]:
ADAPTER = RUN_ROOT_P / TRAIN_RUN_ID / "final_adapter"
print("adapter:", ADAPTER, "(존재:", ADAPTER.exists(), ")")

vqa("predict",
    "--config", CONFIG,
    "--run-id", TRAIN_RUN_ID,
    "--csv", "train.csv",
    "--ids-file", VAL_IDS_FILE,
    "--adapter", str(ADAPTER),
    "--out-name", VAL_OUT_NAME,
    *OVERRIDES)

Path("artifacts/eval").mkdir(parents=True, exist_ok=True)
script("evaluate_run.py",
       "--pred", str(RUN_ROOT_P / TRAIN_RUN_ID / "preds" / (VAL_OUT_NAME + ".csv")),
       "--gold", str(Path(os.environ["VQA_DATA_ROOT"]) / "train.csv"),
       "--val-ids", VAL_IDS_FILE,
       "--json-out", "artifacts/eval/%s_%s.json" % (TRAIN_RUN_ID, VAL_OUT_NAME),
       "--wrong-out", "artifacts/eval/%s_%s_wrong.csv" % (TRAIN_RUN_ID, VAL_OUT_NAME),
       "--note", "9B QLoRA 미세조정 후 validation")

## 7. dev teacher 추론 (의사 라벨 후보 만들기)

미세조정 전 9B base로 `dev.csv` 를 추론합니다. 이 결과와 학생 응답의 다수결이 일치하는 행만
의사 라벨 후보가 됩니다.

계획서 7절의 규칙을 반드시 지켜야 합니다. teacher 프롬프트에는 학생 응답이나 다수결 결과를
절대 넣지 않습니다. 응답을 본 뒤 같은 답을 내도록 유도하면 일치 조건 자체가 의미를 잃습니다.
아래 명령이 `dev.csv` 를 그대로 읽되 `answer1`~`answer5` 컬럼을 입력에 사용하지 않는다는 점은
코어 구현 쪽에서 보장합니다. 여기에서는 `--adapter none` 으로 base 모델임을 명시합니다.

여러 PC로 나눌 때는 `DEV_SHARD` 를 각자 다르게 지정하십시오. GPU를 쓰는 다른 작업과 같은 PC에서
동시에 돌리지 않습니다.

In [ ]:
args = ["--config", CONFIG,
        "--run-id", DEV_TEACHER_RUN_ID,
        "--csv", "dev.csv",
        "--adapter", "none",
        "--out-name", "dev_teacher"] + OVERRIDES
if DEV_SHARD:
    args += ["--shard", DEV_SHARD]
    print("이 PC가 맡은 dev 구간:", DEV_SHARD)

vqa("predict", *args)

print()
print("===== teacher 추론 결과 공유 =====")
print("run_id   : %s" % DEV_TEACHER_RUN_ID)
print("shard    : %s" % (DEV_SHARD or "전체"))
print("출력 폴더: %s" % (RUN_ROOT_P / DEV_TEACHER_RUN_ID / "preds"))
print("다음 담당: 수빈이 다수결과 대조해서 후보를 추리고 100개를 검수합니다.")
print("================================")